# Week 10 — Function 01

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [Model and acquisition](#model-acquisition)
9. [Week 10 proposal](#week-10-proposal)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)

<a id="overview"></a>
## 1. Overview

This focused review mirrors the canonical Week 10 methodology for Function 01, with Weeks 1–9 observed and Week 10 proposed only.

<a id="objectives"></a>
## 2. Objectives

Validate the 2-dimensional evidence, assess the latest returned point, and reproduce the recorded GP-UCB proposal without look-ahead.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

Starter arrays come from `Week_01/Function_nn/03_Data`; exact returned pairs come from `Results/query_output_ledger.csv`. The Week 10 return is excluded because it was unknown when the proposal was selected.

<a id="environment-setup"></a>
## 4. Environment and setup

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
for candidate in (ROOT,*ROOT.parents):
    if (candidate/'Week_10'/'Function_01').is_dir(): ROOT=candidate; break
else: raise FileNotFoundError('Could not locate repository root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from Code.historical_function_review import analyse_historical_function

<a id="data-validation"></a>
## 5. Data validation

In [2]:
observations, summary, proposal, diagnostic_figure = analyse_historical_function(10, 1, ROOT)
input_columns=[f'x{i}' for i in range(1,3)]
inputs=observations[input_columns].to_numpy(float)
outputs=observations['objective'].to_numpy(float)
assert inputs.shape==(19,2) and outputs.shape==(19,)
assert np.isfinite(inputs).all() and np.isfinite(outputs).all()
assert np.all((inputs>=0)&(inputs<=1))
observations

,query,evidence,x1,x2,objective
0,1,starter,0.319404,0.762959,1.322677e-79
1,2,starter,0.574329,0.879898,1.033078e-46
2,3,starter,0.731024,0.733000,7.710875e-16
3,4,starter,0.840353,0.264732,3.341771e-124
4,5,starter,0.650114,0.681526,-3.606063e-03
5,6,starter,0.410437,0.147554,-2.159249e-54
6,7,starter,0.312691,0.078723,-2.089093e-91
7,8,starter,0.683418,0.861057,2.535001e-40
8,9,starter,0.082507,0.403488,3.606771e-81
9,10,starter,0.883890,0.582254,6.229856e-48


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

All comparisons are descriptive and within-function; no causal, global-optimum, or cross-function ranking claim is made.

In [3]:
pd.Series({k:v for k,v in summary.items() if k!='proposal'}, name='verified evidence')

week                                                                            10
function                                                                         1
dimensions                                                                       2
starter_observations                                                            10
recorded_pairs                                                                   9
total_verified_observations                                                     19
best_query                                                                       3
best_input                                [0.7310236309563586, 0.7329998764152272]
best_output                                                                    0.0
latest_verified_query                                                           19
latest_verified_input                                          [0.42141, 0.935804]
latest_verified_output                                                        -0.0
late

<a id="visual-eda"></a>
## 7. Visual EDA

Orange markers are returned Weeks 1–9 observations; the star is the verified incumbent. The Week 10 proposal is deliberately absent.

In [4]:
display(diagnostic_figure)
plt.close(diagnostic_figure)

<Figure size 1200x450 with 3 Axes>

<a id="model-acquisition"></a>
## 8. Model and acquisition

Method: **GP-UCB**. This adaptive policy is a heuristic chosen from the evidence available at the decision boundary; it is not a statistically controlled acquisition comparison.

<a id="week-10-proposal"></a>
## 9. Week 10 proposal

Proposed only: `[0.379403, 0.071186]`. Decision record: Chosen using evidence through Week 9, before the Week 10 return.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

In [5]:
candidate=np.asarray(proposal['query'],dtype=float)
assert proposal['status']=='proposed_only'
assert candidate.shape==(2,) and np.all((candidate>=0)&(candidate<=0.999999))
duplicate=bool(np.any(np.all(np.isclose(inputs,candidate,rtol=0,atol=5e-7),axis=1)))
assert duplicate==proposal['duplicates_observed_evidence']
assert summary['recorded_pairs']==9
portal='-'.join(f'{value:.6f}' for value in candidate)
assert all(len(part.split('.')[-1])==6 for part in portal.split('-'))
print('Function 01 Week 10 checks passed:', portal, 'duplicate:', duplicate)

Function 01 Week 10 checks passed: 0.379403-0.071186 duplicate: False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

The evidence boundary is locked at 19 verified observations. The Week 10 proposal remains unobserved until its authoritative return is appended at the next checkpoint.